# Forecasting CPI spikes in Gaza
**Misk Elayan · Asil Khalil · Sandra Shwamreh**  
Original Data Analytics for Business group project; repository implementation and continued development by Misk Elayan.

This notebook runs the frozen PCBS analysis end to end. The question is whether previous monthly changes help predict a **next-month expenditure-group increase of at least 10%**. All displayed numerical results are executed outputs. See [methodology](../docs/methodology.md) for choices, timing and limitations.

The supplied original `Copy_of_gaza_cpi_final.ipynb` informs the descriptive scope. See [original notebook review](../docs/original_notebook_review.md) for the corrections to grouping, target timing and evaluation.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import run, read_data, model_frame, FEATURES, TEST_START
from src.models import time_based_split
raw = read_data(ROOT)
result = run(ROOT)
TABLES = ROOT / 'reports/tables'


 threshold_pct               model  train_rows  train_spikes   n  actual_spikes  predicted_spikes  accuracy  precision  recall       f1  tn  fp  fn  tp
          10.0            No spike         260            30 156             25                 0  0.839744   0.000000    0.00 0.000000 131   0  25   0
          10.0         Persistence         260            30 156             25                25  0.833333   0.480000    0.48 0.480000 118  13  13  12
          10.0 Logistic Regression         260            30 156             25                59  0.730769   0.355932    0.84 0.500000  93  38   4  21
          10.0       Random Forest         260            30 156             25                68  0.685897   0.323529    0.88 0.473118  85  46   3  22


## 1. Data overview and validation
The frozen original project pack covers December 2022–February 2026. December is the base observation for January's change. The official workbook now has later months, excluded here. Code strings identify series; English labels can repeat.

All 585 major-group levels and 5,070 detailed levels were compared with the official workbook. Recover the major table's 15 missing February 2024 changes from levels; retain original reported values. The detailed table has 130 codes but 124 English labels.

In [2]:
display(pd.read_csv(ROOT / 'data/processed/source_inventory.csv'))
display(raw.head())
provenance = json.loads((ROOT / 'data/processed/provenance.json').read_text())
display(pd.Series(provenance['major_official_overlap']))
recovered = raw[raw.reported_pct_change.isna() & raw['pct_change'].notna()]
display(recovered[['code', 'date', 'cpi_index', 'pct_change']])
assert len(recovered) == 15
assert raw[['date', 'code', 'cpi_index']].notna().all().all()
assert not raw.duplicated(['code', 'date']).any()

,sheet,rows,codes,unique_english_labels,start,end,missing_levels,missing_reported_changes,recovered_changes,max_reported_change_difference_pp
0,major,585,15,15,2022-12-01,2026-02-01,0,30,15,1.278977e-13
1,detailed,5070,130,124,2022-12-01,2026-02-01,0,130,0,2.273737e-13


,code,group_en,date,cpi_index,reported_pct_change,pct_change
0,01,Food and Non-Alcoholic Beverages,2022-12-01,105.674266,NaN,NaN
1,01,Food and Non-Alcoholic Beverages,2023-01-01,103.224577,-2.318151,-2.318151
2,01,Food and Non-Alcoholic Beverages,2023-02-01,105.058232,1.776375,1.776375
3,01,Food and Non-Alcoholic Beverages,2023-03-01,106.843702,1.699504,1.699504
4,01,Food and Non-Alcoholic Beverages,2023-04-01,107.415250,0.534938,0.534938


rows                                                                           585
codes                            [01, 02, 03, 04, 05, 06, 07, 08, 09, 0999, 10,...
max_absolute_index_difference                                                  0.0
dtype: object

,code,date,cpi_index,pct_change
14,01,2024-02-01,288.244257,28.283217
53,02,2024-02-01,893.033730,227.847456
92,03,2024-02-01,104.910735,1.628192
131,04,2024-02-01,134.598655,2.235667
170,05,2024-02-01,116.210206,3.754116
209,06,2024-02-01,120.460429,6.893013
248,07,2024-02-01,402.293852,6.075064
287,08,2024-02-01,101.593233,0.000000
326,09,2024-02-01,129.303010,0.000000
365,0999,2024-02-01,226.523875,27.254527


## 2. Exploratory analysis
Inspect heterogeneity, large movements and flat reported series. All-items CPI is contextual; it is not an additional independent training group.

![CPI overview](../reports/figures/cpi_overview.png)
![Monthly changes](../reports/figures/monthly_changes.png)

In [3]:
display(pd.read_csv(TABLES / 'group_overview.csv', dtype={'code': str}))
display(pd.read_csv(TABLES / 'largest_increases.csv', dtype={'code': str}).head(8))
print('February 2026 all-items change (%):', result['summary']['aggregate_feb2026_change_pct'])
# Detailed codes are descriptive: parents and children overlap.
display(pd.read_csv(TABLES / 'detailed_group_overview.csv', dtype={'code': str}).head(15))


,code,group_en,months,min_index,max_index,mean_monthly_change,sd_monthly_change,unchanged_months,spikes_10pct
0,01,Food and Non-Alcoholic Beverages,39,103.224577,1468.467915,5.879375,31.419922,0,11
1,02,"Alcholoic Beverages, Tobacco and Narcotics",39,111.018499,8549.875459,27.205461,74.907392,1,16
2,03,Clothing and Footwear,39,97.039966,208.916372,2.252813,13.506895,4,3
3,04,"Housing, Water, Electricity, Gas and Other Fuels",39,106.419269,1405.677150,10.732599,48.145045,0,8
4,05,"Furnishings, Household Equipment and Routine H...",39,100.002068,196.815969,0.782383,10.675996,1,5
5,06,Health,39,87.661755,183.881923,2.000595,3.021346,16,0
6,07,Transport,39,99.134617,755.230862,4.629038,25.622398,0,10
7,08,Information and Communication,39,99.781539,102.406158,-0.035960,0.462224,25,0
8,09,"Recreation, Sport, Culture, Gardens and Pets",39,117.045301,132.203998,0.300560,1.515125,27,0
9,0999,All items of consumer price index,39,102.880413,824.697329,5.540988,24.363152,0,14


,date,code,group_en,pct_change
0,2026-02-01,02,"Alcholoic Beverages, Tobacco and Narcotics",282.774940
1,2024-02-01,02,"Alcholoic Beverages, Tobacco and Narcotics",227.847456
2,2025-03-01,04,"Housing, Water, Electricity, Gas and Other Fuels",207.489928
3,2025-04-01,02,"Alcholoic Beverages, Tobacco and Narcotics",143.260570
4,2023-11-01,02,"Alcholoic Beverages, Tobacco and Narcotics",125.893814
5,2023-10-01,07,Transport,114.097918
6,2024-03-01,02,"Alcholoic Beverages, Tobacco and Narcotics",112.379794
7,2025-04-01,04,"Housing, Water, Electricity, Gas and Other Fuels",110.823997


February 2026 all-items change (%): 37.9227885401


,code,group_en,months,mean_monthly_change,sd_monthly_change,max_monthly_change
0,011613,Dates,39,35.561574,188.246251,1080.327869
1,04522,Liquefied hydrocarbons,39,49.872981,181.131161,925.225428
2,0452,Gas,39,49.872981,181.131161,925.225428
3,01133,Fish preparations,39,20.807373,148.206391,885.766449
4,011331,"Tunas, skipjack or stripe-bellied bonito, prep...",39,20.807373,148.206391,885.766449
5,011631,Apples,39,29.478299,140.487621,742.105263
6,011770.4,Dry garlic,39,36.875620,139.768714,608.620690
7,011931,Salt,39,28.515886,135.942312,680.000000
8,011659,"Other fruits, fresh",39,28.140219,128.766106,575.000000
9,011645,Strawberries,39,21.962494,115.034957,636.616876


## 3. Features, label and chronology
Primary spike: monthly increase ≥10%; sensitivity: 5% and 20%. This interpretable cutoff is a study choice, not an official definition or a test-optimized threshold.

Model codes 01–13 only; exclude 0999 and the overlapping 12+13 aggregate. Lag changes by 1/2/3 months; shift full 3/6-month means and standard deviations; add calendar sine/cosine and group identity. No current-month outcome enters predictors. A complete six-month history is required.

Train July 2023–February 2025. Hold out March 2025–February 2026. Each held-out month uses earlier observed months, including earlier test outcomes, for its lags; preprocessing and fitted models stay frozen. This is sequential one-step evaluation, not a 12-step forecast.

In [4]:
panel = model_frame(raw)
train, test = time_based_split(panel, TEST_START)
display(pd.DataFrame([
    {'segment': name, 'rows': len(d), 'months': d.date.nunique(),
     'start': d.date.min(), 'end': d.date.max(), 'spikes_10pct': int((d['pct_change'] >= 10).sum())}
    for name, d in [('Train', train), ('Test', test)]
]))
display(panel[['date', 'code'] + [f for f in FEATURES if f != 'code']].head())
assert train.date.max() < test.date.min()
assert not set(train.date).intersection(set(test.date))

,segment,rows,months,start,end,spikes_10pct
0,Train,260,20,2023-07-01,2025-02-01,30
1,Test,156,12,2025-03-01,2026-02-01,25


,date,code,pct_change_lag_1,pct_change_lag_2,pct_change_lag_3,pct_change_rolling_mean_3,pct_change_rolling_std_3,pct_change_rolling_mean_6,pct_change_rolling_std_6,month_sin,month_cos
0,2023-07-01,01,-1.083234,-1.074900,0.534938,-0.541065,0.931856,-0.077578,1.673111,-0.5,-0.866025
1,2023-07-01,02,0.680541,0.008699,0.115549,0.268263,0.361018,0.223063,0.492593,-0.5,-0.866025
2,2023-07-01,03,0.249178,0.023877,0.726728,0.333261,0.358891,0.153062,0.654123,-0.5,-0.866025
3,2023-07-01,04,-0.413278,-0.056079,0.255455,-0.071300,0.334626,-0.247981,0.646409,-0.5,-0.866025
4,2023-07-01,05,-0.445403,0.225658,0.057309,-0.054145,0.349138,0.047187,0.547685,-0.5,-0.866025


## 4. Model comparison
Compare no-spike and persistence rules with class-balanced Logistic Regression and a regularized Random Forest (300 trees, depth 4, minimum leaf 5). Settings are fixed, seed 42, probability cutoff 0.5, and scaling/encoding fit on training only. Zero predicted positives gives precision/F1 0, with counts shown.

![Confusion matrices](../reports/figures/confusion_matrices.png)

In [5]:
display(result['metrics'].round(4))
# Independently recompute every primary confusion matrix and metric from predictions.
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
predictions = pd.read_csv(TABLES / 'predictions.csv')
for _, row in result['metrics'].iterrows():
    p = predictions[(predictions.threshold_pct == 10) & (predictions.model == row.model)]
    y, pred = p.is_spike.astype(int), p.prediction.astype(int)
    np.testing.assert_array_equal(confusion_matrix(y, pred, labels=[0,1]).ravel(), row[['tn','fp','fn','tp']].to_numpy(dtype=int))
    for name, function in [('accuracy',accuracy_score),('precision',precision_score),('recall',recall_score),('f1',f1_score)]:
        kwargs = {} if name == 'accuracy' else {'zero_division': 0}
        assert np.isclose(function(y, pred, **kwargs), row[name])
print('All primary metrics reconcile with saved predictions.')

,threshold_pct,model,train_rows,train_spikes,n,actual_spikes,predicted_spikes,accuracy,precision,recall,f1,tn,fp,fn,tp
4,10.0,No spike,260,30,156,25,0,0.8397,0.0000,0.00,0.0000,131,0,25,0
5,10.0,Persistence,260,30,156,25,25,0.8333,0.4800,0.48,0.4800,118,13,13,12
6,10.0,Logistic Regression,260,30,156,25,59,0.7308,0.3559,0.84,0.5000,93,38,4,21
7,10.0,Random Forest,260,30,156,25,68,0.6859,0.3235,0.88,0.4731,85,46,3,22


All primary metrics reconcile with saved predictions.


## 5. Sensitivity, temporal robustness and uncertainty
These thresholds define different tasks; do not choose one because it scores better. Earlier validation folds stay inside final training dates. Descriptive F1 intervals resample whole test months (1,000 draws), preserving same-month group dependence but not all serial dependence.

![Threshold sensitivity](../reports/figures/sensitivity.png)

In [6]:
display(result['sensitivity'][['threshold_pct','model','actual_spikes','accuracy','precision','recall','f1']].round(4))
display(pd.read_csv(TABLES / 'temporal_validation.csv')[['validation_start','model','train_rows','actual_spikes','precision','recall','f1']].round(4))
display(pd.read_csv(TABLES / 'f1_month_bootstrap.csv').round(4))

,threshold_pct,model,actual_spikes,accuracy,precision,recall,f1
0,5.0,No spike,30,0.8077,0.0000,0.0000,0.0000
1,5.0,Persistence,30,0.8462,0.6000,0.6000,0.6000
2,5.0,Logistic Regression,30,0.7051,0.3710,0.7667,0.5000
3,5.0,Random Forest,30,0.7115,0.3913,0.9000,0.5455
4,10.0,No spike,25,0.8397,0.0000,0.0000,0.0000
5,10.0,Persistence,25,0.8333,0.4800,0.4800,0.4800
6,10.0,Logistic Regression,25,0.7308,0.3559,0.8400,0.5000
7,10.0,Random Forest,25,0.6859,0.3235,0.8800,0.4731
8,20.0,No spike,19,0.8782,0.0000,0.0000,0.0000
9,20.0,Persistence,19,0.8654,0.4444,0.4211,0.4324


,validation_start,model,train_rows,actual_spikes,precision,recall,f1
0,2024-03-01,No spike,104,12,0.0000,0.0000,0.0000
1,2024-03-01,Persistence,104,12,0.5385,0.5833,0.5600
2,2024-03-01,Logistic Regression,104,12,0.5000,0.2500,0.3333
3,2024-03-01,Random Forest,104,12,0.3030,0.8333,0.4444
4,2024-09-01,No spike,182,7,0.0000,0.0000,0.0000
5,2024-09-01,Persistence,182,7,0.2857,0.2857,0.2857
6,2024-09-01,Logistic Regression,182,7,0.2069,0.8571,0.3333
7,2024-09-01,Random Forest,182,7,0.1923,0.7143,0.3030


,model,f1_p025,f1_p975,resamples,resampling_unit
0,Logistic Regression,0.3467,0.6452,1000,month
1,No spike,0.0000,0.0000,1000,month
2,Persistence,0.2941,0.6207,1000,month
3,Random Forest,0.3095,0.6061,1000,month


## 6. Interpretation and limits
At the primary threshold, Logistic Regression detects 21/25 spikes with 38 false alarms; Random Forest detects 22/25 with 46 false alarms. Their F1 scores (0.500 and 0.473) are close to persistence (0.480), and no model wins consistently across thresholds. High recall does not make these reliable alerts.

Only 20 modeled training months and 12 test months are available. Groups share shocks; source coverage and flat reported series limit interpretation. Probabilities are not calibrated. Historical release delays/revisions are not simulated. The regional price snapshot is not a monthly panel. No causal or policy claim is warranted.

ARIMA/SARIMA is deferred because 26 available training changes, flat series and abrupt regime changes make a seasonal benchmark fragile. Persistence provides a directly comparable time-series baseline.

Data: **Palestinian Central Bureau of Statistics (PCBS)**, [official dataset](https://data.humdata.org/dataset/0e06dbe6-8eeb-4b26-ba12-652520b44177), source CC BY designation. [Data provenance](../data/README.md) and [full methodology](../docs/methodology.md).